# FAST-002 — minimal targeted repair
Attach original FAST-001 Save Version Output via Add Input. Enable Internet, one GPU and existing GITHUB_TOKEN. Run All explicitly requests Train-only calibration and, only if all 20 contexts pass, at most 20 B01/B11 paths ×500 steps. A/B00/B10 are never retrained. Fresh Confirm does not select scales. Missing weights or failed calibration stops and retains evidence; reports automatically return to GitHub. No HF_TOKEN, Kaggle API token, version number or manual Seed/hash entry. This is targeted repair, not independent replication.


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

SOURCE_SHA = 'afc0032a3adad7008980f89bac44005447338274'
BASE_REL = 'research/projects/cellular-language-models/experiments/clm-002-fast-002'
PATTERNS = ['/pyproject.toml', '/src/', '/shared/python/', '/research/projects/cellular-language-models/engineering/kaggle-continuation/run.py', '/research/projects/cellular-language-models/experiments/006-path-first-clm/', '/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/', '/research/projects/continual-learning/context/04-continual-learning/scripts/research/', '/research/projects/cellular-language-models/context/08-hybrid-clm/scripts/research/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r2/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3b/run.py', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3b/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3b/config/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3c/moe-bridge/run.py', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3c/moe-bridge/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3c/moe-bridge/config/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3d/run.py', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3d/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3d/config/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3e/run.py', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3e/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3e/config/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3f-connection-admission/run.py', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3f-connection-admission/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3f-connection-admission/config/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3f-v2/run.py', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3f-v2/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3f-v2/config/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3g/run.py', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3g/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3g/config/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3h/run.py', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3h/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3h/config/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3i/run.py', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3i/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3i/config/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3j/run.py', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3j/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3j/config/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3k/run.py', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3k/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3k/config/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3l/run.py', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3l/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3l/config/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m/run.py', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m/config/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m-e/run.py', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m-e/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m-e/config/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m-f-d1/run.py', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m-f-d1/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m-f-d1/config/', '/research/projects/cellular-language-models/experiments/clm-002-cell-differentiation-joint-value/run.py', '/research/projects/cellular-language-models/experiments/clm-002-cell-differentiation-joint-value/implementation/', '/research/projects/cellular-language-models/experiments/clm-002-cell-differentiation-joint-value/config/', '/research/projects/cellular-language-models/experiments/jct-001/run.py', '/research/projects/cellular-language-models/experiments/jct-001/implementation/', '/research/projects/cellular-language-models/experiments/jct-001/config/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m/registration/artifact-recovery-001/independent-review-002/tokenizer.json', '/research/projects/cellular-language-models/experiments/clm-002-fast-001/', '!**/reports/**', '!**/artifacts/**', '!/research/projects/cellular-language-models/experiments/clm-002-fast-001/registration/data-manifests/', '!/research/projects/cellular-language-models/experiments/clm-002-fast-001/verification/', '/research/projects/cellular-language-models/experiments/clm-002-fast-002/', '!/research/projects/cellular-language-models/experiments/clm-002-fast-002/verification/', '!/research/projects/cellular-language-models/experiments/clm-002-fast-002/registration/Story-confirm.json', '!/research/projects/cellular-language-models/experiments/clm-002-fast-002/registration/Arithmetic-confirm.json', '!/research/projects/cellular-language-models/experiments/clm-002-fast-002/registration/HISTORICAL_AUDIT.json']
CODE = Path('/kaggle/working') / ('clm002-fast002-code-' + SOURCE_SHA[:12])
OUTPUT = Path('/kaggle/working/clm002-fast002-output')
ASSETS = Path('/kaggle/working/clm002-fast002-assets')
print('[FAST002] Run All requests train-only calibration followed by conditional targeted repair.', flush=True)
print('[FAST002] Attach original FAST-001 Save Version Output as Input. No old baseline retraining.', flush=True)
print('[FAST002] Calibration <=160 candidate optimizer calls, main updates=0. Only if passed: <=20 B01/B11 jobs x500 main steps. Thresholds unchanged.', flush=True)

def run(args, label, timeout=None):
    print('[FAST002]', label, flush=True)
    return subprocess.run(list(map(str, args)), check=True, timeout=timeout)

from kaggle_secrets import UserSecretsClient
try:
    token = UserSecretsClient().get_secret('GITHUB_TOKEN').strip()
except Exception:
    raise RuntimeError('Enable existing GITHUB_TOKEN for automatic verified report return') from None
if not token:
    raise RuntimeError('Empty GITHUB_TOKEN')
os.environ['GITHUB_TOKEN'] = token
if importlib.util.find_spec('tokenizers') is None:
    run([sys.executable, '-m', 'pip', 'install', '--quiet', 'tokenizers==0.20.3'], 'Prepare missing tokenizer runtime', 60)
if not (CODE / '.git').exists():
    CODE.mkdir(parents=True, exist_ok=True)
    run(['git', 'init', '--quiet', CODE], 'Initialize pinned runtime')
head = subprocess.run(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], capture_output=True, text=True)
if head.returncode:
    run(['git', '-C', CODE, 'config', 'remote.origin.url', 'https://github.com/ArcheLabs/mini-cells.git'], 'Set source')
    run(['git', '-C', CODE, 'config', 'remote.origin.promisor', 'true'], 'Use partial clone')
    run(['git', '-C', CODE, 'config', 'remote.origin.partialclonefilter', 'blob:none'], 'Exclude unrelated blobs')
    run(['git', '-C', CODE, 'sparse-checkout', 'init', '--no-cone'], 'Prepare runtime checkout')
    run(['git', '-C', CODE, 'sparse-checkout', 'set', '--no-cone', *PATTERNS], 'Select runtime files')
    run(['git', '-C', CODE, 'fetch', '--depth=1', '--filter=blob:none', 'origin', SOURCE_SHA], 'Fetch fixed code', 60)
    run(['git', '-C', CODE, 'checkout', '--detach', 'FETCH_HEAD'], 'Load implementation', 60)
actual = subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], text=True).strip()
if actual != SOURCE_SHA:
    raise RuntimeError('FAST002_SOURCE_MISMATCH')
try:
    run([sys.executable, CODE / BASE_REL / 'run.py', '--output', OUTPUT, '--assets', ASSETS,
         '--run-repair', '--publish-github'], 'Audit -> checkpoint verification -> Train calibration -> conditional repair -> GitHub return')
finally:
    print('Save Version retains reports and new private checkpoints:', OUTPUT, flush=True)
    print('Private historical weights are read from Input and never pushed to GitHub.', flush=True)
    print('Publication retry reuses completed results; never repeat training for network failures.', flush=True)
